# 2.2 Entrenamiento y ajuste: corridas en Colab

Entrena las corridas de la Actividad 2.2 (`oe2/docs/plan_oe2.md`) con `comun_oe2.entrenar`: YOLOv11n o YOLOv11s preentrenado en COCO, 1280 px, lote 16. Cada corrida se evalúa en los granos de su validación, cuenta los falsos positivos sobre los fondos de medición y añade una fila a `oe2/resultados/experimentos/<fase>.csv`.

| Fase | Qué entrena | Corridas | Tiempo aproximado (A100) |
| :--- | :--- | :---: | :--- |
| `fondos` | Referencia con y sin fondos negativos, 2 modelos × 3 semillas (13, 29, 51) | 12 | ~10 min por corrida |
| `cribado` | Optimizador → tasa de aprendizaje → aumento, por modelo, 3 semillas; la referencia viene de `fondos` | hasta 42 | ~3 h por modelo |
| `confirmacion` | Ganadora del cribado frente a la referencia en los pliegues 0, 1 y 3 × 3 semillas (solo si hay ganadora) | 0 a 18 por modelo | ~3 h con aumento A; ~4 h si la ganadora usa B o C |
| `final` | Configuración adoptada, 200 épocas con `patience = 50`, semillas 42, 7 y 28 | 6 | ~15 min por corrida |

Cada fase usa las decisiones de las anteriores, que se calculan a partir del registro con las reglas fijadas antes de entrenar.

**Uso:**
1. Subir `oe2_nube.zip` a la raíz de Google Drive (uno solo) y abrir este notebook con una **A100**.
2. Elegir `FASE`, `MODELOS_SESION` y, en `fondos`, `confirmacion` y `final`, `SEMILLAS_SESION`. El cribado entrena las 3 semillas de cada paso antes de decidir el siguiente, así que se reparte solo por modelo. Para repartir una fase en varias sesiones a la vez, se usa una copia del notebook por sesión.
3. Ejecutar todo. Al terminar, la sesión descarga `oe2_<fase>_<modelos>_s<semillas>.zip`.
4. **Fase siguiente:** subir a la raíz de Drive los zips descargados de las fases anteriores, sin cambiarles el nombre. Al iniciar se leen sus registros, de modo que las decisiones y las corridas ya hechas se recuperan sin volver a entrenar. Solo se leen los zips con el nombre de las descargas de este notebook (`oe2_<fase>_<modelos>_s<semillas>.zip`); cualquier otro `oe2_*.zip` se ignora y se avisa.

En local, `oe2/2.2_entrenamiento.ipynb` incorpora las descargas y analiza cada fase.

In [ ]:
FASE = 'fondos'                     # 'fondos', 'cribado', 'confirmacion' o 'final'
MODELOS_SESION = ('yolo11n', 'yolo11s')
SEMILLAS_SESION = (13, 29, 51)      # fondos y confirmacion: 13, 29, 51; final: 42, 7, 28 (el cribado usa siempre las 3 de selección)

In [ ]:
# --- Entorno: local o Colab ---
import os, re, sys, glob, json, shutil, subprocess, time, zipfile
from pathlib import Path

os.environ['YOLO_VERBOSE'] = 'False'   # sin barras de progreso: la interfaz de Colab se congela con tanta salida
PLATAFORMA = 'colab' if 'COLAB_RELEASE_TAG' in os.environ else 'local'
if PLATAFORMA == 'local':   # este notebook vive en nube/; el código del OE2, en oe2/
    RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'oe2' / 'comun_oe2.py').exists()) / 'oe2'
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.4.174', 'albumentations==2.0.8'], check=True)
    from google.colab import drive
    drive.mount('/content/drive')   # el paquete está en Google Drive, que solo se lee
    base, origen = Path('/content'), '/content/drive/MyDrive'
    if not glob.glob(str(base / '*' / 'oe2' / 'comun_oe2.py')):
        paquete = glob.glob(f'{origen}/oe2_nube*.zip') or glob.glob(f'{origen}/**/oe2_nube*.zip', recursive=True)
        assert paquete, f"No se encontró oe2_nube.zip en Drive. Zips en la raíz: {glob.glob(f'{origen}/*.zip')}"
        zipfile.ZipFile(paquete[0]).extractall(base)
    RAIZ = Path(glob.glob(str(base / '*' / 'oe2' / 'comun_oe2.py'))[0]).parent
os.chdir(RAIZ); sys.path.insert(0, str(RAIZ))

import pandas as pd
import torch
from IPython.display import display
import comun_oe2 as E

if isinstance(MODELOS_SESION, str):   # ('yolo11s') sin coma es un texto, no una tupla
    MODELOS_SESION = (MODELOS_SESION,)
if isinstance(SEMILLAS_SESION, int):  # (42) sin coma es un número, no una tupla
    SEMILLAS_SESION = (SEMILLAS_SESION,)
assert FASE in ('fondos', 'cribado', 'confirmacion', 'final'), FASE
assert set(MODELOS_SESION) <= set(E.MODELOS), f'MODELOS_SESION debe contener {E.MODELOS}'
semillas_fase = E.SEMILLAS_FINAL if FASE == 'final' else E.SEMILLAS_SELECCION
assert set(SEMILLAS_SESION) <= set(semillas_fase), f'En la fase {FASE}, SEMILLAS_SESION debe contener {semillas_fase}'

if PLATAFORMA == 'colab':
    # Registros de las fases anteriores: los zips descargados de este notebook y subidos a la raíz de Drive
    # (oe2_<fase>_<modelos>_s<semillas>.zip; Drive puede añadir ' (1)' a una copia). Cualquier otro zip se ignora
    descarga = re.compile(r'^oe2_(fondos|cribado|confirmacion|final)_(n|s|n_s)_s\d+(_\d+)*( ?\(\d+\))?\.zip$')
    for previo in sorted(glob.glob(f'{origen}/oe2_*.zip')):
        if not descarga.match(Path(previo).name):
            if not Path(previo).name.startswith('oe2_nube'):
                print('Ignorado (no es una descarga de este notebook):', Path(previo).name)
            continue
        with zipfile.ZipFile(previo) as z:
            for n in z.namelist():
                if n.startswith('oe2/resultados/experimentos/') and n.endswith('.csv'):
                    import io
                    E.anotar_registro(E.REGISTRO, Path(n).stem, pd.read_csv(io.BytesIO(z.read(n))).to_dict('records'))
        print('Registro restaurado:', Path(previo).name)
    E.registrar_entorno(RAIZ / 'resultados')
assert torch.cuda.is_available(), 'No hay GPU: Entorno de ejecución > Cambiar tipo de entorno > A100'
GPU = torch.cuda.get_device_name(0)
print('Plataforma:', PLATAFORMA, '| GPU:', GPU, '| fase:', FASE, '| modelos:', MODELOS_SESION, '| semillas:', SEMILLAS_SESION)
if PLATAFORMA == 'colab' and 'A100' not in GPU:
    print('AVISO: todas las corridas deben usar la misma GPU (A100).')
print(f"{len(E.leer_registro())} corridas en el registro")

## 1. Plan de la sesión

Corridas de la fase que faltan en el registro. En local no se entrena: solo se muestra el plan y el estado de las decisiones.

In [ ]:
hechas = set(E.leer_registro()['id'])
inicio = {'t0': time.time(), 'n': 0}
ids_sesion = set()

def entrenar(modelo, cfg, semilla, particion):
    # Entrena una corrida que falta (solo en Colab) e informa del avance
    rid = E.id_corrida(modelo, cfg, semilla, particion)
    ids_sesion.add(rid)
    if PLATAFORMA != 'colab':
        raise RuntimeError(f'Falta {rid}: en local no se entrena')
    print(f'[{FASE}] {rid}')
    f = E.entrenar(modelo, cfg, semilla, particion, FASE, E.RUNS, E.REGISTRO, E.TRABAJO)
    inicio['n'] += 1
    print(f"   mAP50-95 {f['mAP50_95']:.4f} | FP impurezas {f.get('fp_impurezas', float('nan')):.2f} por imagen | "
          f"{f['epocas_ejecutadas']} épocas | {f['minutos']} min | {(time.time() - inicio['t0']) / 3600:.1f} h en la sesión")
    return f

def plan_fondos():
    return [(m, dict(E.REFERENCIA, fondos=f), s, 'valid') for m in MODELOS_SESION for s in SEMILLAS_SESION for f in (False, True)]

def decision_fondos():
    tabla, dec = E.decidir_fondos(E.REGISTRO)
    display(tabla.round(4))
    print('Decisión de fondos:', dec if dec else 'pendiente (faltan corridas de la fase fondos)')
    return dec

if FASE == 'fondos':
    plan = plan_fondos()
    faltan = [p for p in plan if E.id_corrida(*p) not in hechas]
    print(f'{len(plan)} corridas; {len(faltan)} por entrenar')
elif FASE in ('cribado', 'confirmacion', 'final'):
    dec = decision_fondos()
    assert dec is not None, 'Primero hay que completar la fase fondos (sube sus zips a la raíz de Drive)'

## 2. Corridas

In [ ]:
if FASE == 'fondos':
    for p in plan:
        if E.id_corrida(*p) in hechas:
            ids_sesion.add(E.id_corrida(*p))
        elif PLATAFORMA == 'colab':
            entrenar(*p)
    decision_fondos()

elif FASE == 'cribado':
    for modelo in MODELOS_SESION:
        tabla, ganadora = E.cribado(modelo, E.REGISTRO, dec['adoptar'], entrenar if PLATAFORMA == 'colab' else None)
        print(f'\n{modelo}: cribado', 'completo' if ganadora else 'incompleto', '| ganadora:', ganadora)
        display(tabla.round(4))

elif FASE == 'confirmacion':
    for modelo in MODELOS_SESION:
        _, ganadora = E.cribado(modelo, E.REGISTRO, dec['adoptar'])
        assert ganadora is not None, f'Falta completar el cribado de {modelo}'
        ref = dict(E.REFERENCIA, fondos=dec['adoptar'])
        if ganadora == ref:
            print(f'{modelo}: la ganadora del cribado es la referencia; no necesita confirmación')
            continue
        for semilla in SEMILLAS_SESION:
            for particion in E.PLIEGUES_CONFIRMACION:   # sin p2: es la partición del cribado
                for cfg in (ref, ganadora):
                    if E.id_corrida(modelo, cfg, semilla, particion) in hechas:
                        ids_sesion.add(E.id_corrida(modelo, cfg, semilla, particion))
                    elif PLATAFORMA == 'colab':
                        entrenar(modelo, cfg, semilla, particion)
        tabla, adoptada = E.confirmacion(modelo, ganadora, E.REGISTRO, dec['adoptar'])
        print(f'{modelo}: confirmación', 'completa' if adoptada else 'incompleta', '| adoptada:', adoptada, tabla.attrs)
        display(tabla.round(4))

elif FASE == 'final':
    adoptada, _ = E.configuracion_ajustada(E.REGISTRO)
    for modelo in MODELOS_SESION:
        assert adoptada[modelo] is not None, f'Faltan corridas del cribado o de la confirmación de {modelo}'
        print(f'{modelo}: configuración adoptada {adoptada[modelo]}')
        for semilla in SEMILLAS_SESION:
            if E.id_corrida(modelo, adoptada[modelo], semilla, 'final') in hechas:
                ids_sesion.add(E.id_corrida(modelo, adoptada[modelo], semilla, 'final'))
            elif PLATAFORMA == 'colab':
                entrenar(modelo, adoptada[modelo], semilla, 'final')

## 3. Descarga

Empaqueta el registro de la fase, el entorno y las carpetas de las corridas de esta sesión en `oe2_<fase>_<modelos>_s<semillas>.zip` y lo descarga.

In [ ]:
if PLATAFORMA == 'colab':
    sufijo = '_'.join(m.replace('yolo11', '') for m in MODELOS_SESION)
    semillas = '_'.join(map(str, E.SEMILLAS_SELECCION if FASE == 'cribado' else SEMILLAS_SESION))
    destino = Path('/content') / f'oe2_{FASE}_{sufijo}_s{semillas}.zip'
    with zipfile.ZipFile(destino, 'w', zipfile.ZIP_DEFLATED) as z:
        for f in [*E.REGISTRO.glob('*.csv'), *(RAIZ / 'resultados').glob('entorno_*.json')]:
            z.write(f, f.relative_to(E.REPO))
        for f in E.RUNS.rglob('*'):
            if f.is_file() and f.relative_to(E.RUNS).parts[0] in ids_sesion:   # los .pt ya están comprimidos
                z.write(f, f.relative_to(E.REPO), compress_type=zipfile.ZIP_STORED if f.suffix == '.pt' else zipfile.ZIP_DEFLATED)
    print(f'{destino} ({destino.stat().st_size / 1e6:.0f} MB, {len(ids_sesion)} corridas)')
    from google.colab import files
    files.download(str(destino))